# TALC demo

Load a pretrained TALC model, encode an audio sample into latents, and decode them back.

Install the extras first with `pip install -e ".[notebook]"`. Model weights are downloaded from the Hugging Face Hub on first use and cached.

In [ ]:
import librosa
from IPython.display import Audio, display

from talc import TALC

## Load the model

Three checkpoints are released: `'talc'` (the proposed model), `'isotropic'` and `'baseline'`. To use a local checkpoint instead, pass its path.

In [ ]:
model = TALC(variant='talc')
# model = TALC('path/to/talc.pt')  # local checkpoint

SR = model.sample_rate

## Load an audio sample

The demo uses librosa's piano example (The Piano Lady, *Pistachio Ice Cream Ragtime*), downloaded on first use. Set `AUDIO_PATH` to any local file to use your own; `load_audio` downmixes it to mono and resamples it to 44.1 kHz.

In [ ]:
AUDIO_PATH = librosa.example('pistachio')

wav = model.load_audio(AUDIO_PATH)[:30 * SR]  # first 30 s
print(f'{len(wav) / SR:.1f} s')
display(Audio(wav.numpy(), rate=SR))

## Encode

`encode` returns the latent sequence: 64 channels at about 10.8 frames per second. Audio is processed in 60-second chunks, and a sample shorter than that in one piece; pass `chunk_sec=` to both `encode` and `decode` to change the chunk size.

In [ ]:
latents = model.encode(wav)
print(f'Latents: {latents.shape[0]} channels x {latents.shape[1]} frames')

## Decode

`decode` also needs the number of encoded samples, which tells it how the audio was chunked.

In [ ]:
decoded = model.decode(latents, len(wav))
display(Audio(decoded.numpy(), rate=SR))